In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

device = 'cuda' if torch.cuda.is_available() else 'cpu'

## Hyperparameters

The current set of hyperparameters for our model implementation is listed below:

In [2]:
# new hyperparameters
embedding_size = 32          # "n_embd" ... size of the embedding tensors
head_size = 16               # size of a single head of attention

# hyperparameters
batch_size = 32              # how many independent sequences to parallel-process?
max_context_size = 8         # "block_size" ... maximum context length for predictions
max_iters = 5000
eval_interval = 500
learning_rate = 1e-3
loss_estimation_iters = 200  # "eval_iters" ... num. iters for estimate_loss

----

## "Magical" Helper

In [3]:
from helper import *

def get_batch(split):
    data = training_data if split == 'train' else validation_data
    ix = torch.randint(len(data) - max_context_size, (batch_size,))
    x = torch.stack([data[i:i+max_context_size] for i in ix]).to(device)
    y = torch.stack([data[i+1:i+max_context_size+1] for i in ix]).to(device)
    return x,y

----

## The Mathematical Trick in Self-Attention

> Attention is a way for a neural network to decide which parts of the available information are most important for what it is doing right now.

`BigramLanguageModel` is supposed to be a character-based language model that predicts the next character (token) given its preceding sequence of characters (context).

However, our initial implementation of `BigramLanguageModel` does not have any allowance for letting the $t^{\text{th}}$ token of a sequence use any knowledge of its preceding $t-1$ tokens.

For a decoder model like GPT-2 (please see [Language Models are Unsupervised Multitask Learners](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf) ) that predicts the next token given its preceding tokens in the context, _self-attention_ means that the model can only use knowledge of the preceding tokens; there is no way to look at tokens in the future to inform predictions.

Let's try implementing a very simple form of _self-attention_ that just uses the average of the preceding tokens.

In [4]:
torch.manual_seed(1337)

B,T,C = 4,8,2
x = torch.randn(B,T,C)   # batch, time, channels
x.shape

torch.Size([4, 8, 2])

### Version 1: Naive implementation

This is $O(n^2)$, but the general idea is that each row of `xbow` will be the average of all its preceding rows.

In [5]:
# we want x[b,t] = mean_{i<=t} x[b,i]
xbow = torch.zeros((B,T,C))

for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1]   # (t,C)
        xbow[b,t] = torch.mean(xprev, 0) # average calculated over the Time dimension

In [6]:
x[0]

tensor([[ 0.1808, -0.0700],
        [-0.3596, -0.9152],
        [ 0.6258,  0.0255],
        [ 0.9545,  0.0643],
        [ 0.3612,  1.1679],
        [-1.3499, -0.5102],
        [ 0.2360, -0.2398],
        [-0.9211,  1.5433]])

In [7]:
xbow[0]

tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])

### Version 2: matrix multiplication

In [8]:
torch.manual_seed(42)
a = torch.tril(torch.ones(3,3))
a = a / torch.sum(a, 1, keepdim=True)
b = torch.randint(0, 10, (3,2)).float()
c = a @ b
print('a=')
print(a)
print('--')
print('b=')
print(b)
print('--')
print('c=')
print(c)

a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
--
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
--
c=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


In [9]:
wei= torch.tril(torch.ones(T,T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x   # (B,T,T) @ (B,T,C) ----> (B,T,C)

In [10]:
torch.allclose(xbow, xbow2, atol=1e-5)

True

In [11]:
print(xbow[0])
print()
print(xbow2[0])

tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])

tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])


### Version 3: With Softmax

In [12]:
tril = torch.tril(torch.ones(T,T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril==0, float('-inf'))
print(f"before Softmax:\n{wei}")
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
print()
print(f"after Softmax:\n{wei}")
print()
print(torch.allclose(xbow, xbow3, atol=1e-5))

before Softmax:
tensor([[0., -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0., 0., 0., 0.]])

after Softmax:
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 

In [13]:
print(xbow[0])
print()
print(xbow3[0])

tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])

tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])


### Version 4: self-attention!

> Query (Q) — What am I looking for?
> <br/>
> Key (K) — What information do I represent?
> <br/>
> Value (V) — What information should I provide if selected?

... and the "self" in self-attention comes from the fact that the source of query `q`, key `k`, and value `v` all come from the input `x` in our decoder implementation.

In [14]:
torch.manual_seed(1337)

B,T,C = 4,8,32          # batch, time, channels
x = torch.randn(B,T,C)

# let's see a single Head perform self-attention
head_size = 16

key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x)              # (B,T,16)
q = query(x)            # (B,T,16)
wei = q @ k.transpose(-2, -1)  # (B,T,16) @ (B,16,T) ----> (B,T,T)

tril = torch.tril(torch.ones(T,T))

# before causal masking
print(f"before causal masking:\n{wei[0]}\n")
wei = wei.masked_fill(tril==0, float('-inf')) 
print(f"after causal masking:\n{wei[0]}\n")
wei = F.softmax(wei, dim=-1) 
print(f"and after Softmax:\n{wei[0]}\n")

v = value(x)
out = wei @ v

print(f"out.shape: {out.shape}")

before causal masking:
tensor([[-1.7629, -1.3011,  0.5652,  2.1616, -1.0674,  1.9632,  1.0765, -0.4530],
        [-3.3334, -1.6556,  0.1040,  3.3782, -2.1825,  1.0415, -0.0557,  0.2927],
        [-1.0226, -1.2606,  0.0762, -0.3813, -0.9843, -1.4303,  0.0749, -0.9547],
        [ 0.7836, -0.8014, -0.3368, -0.8496, -0.5602, -1.1701, -1.2927, -1.0260],
        [-1.2566,  0.0187, -0.7880, -1.3204,  2.0363,  0.8638,  0.3719,  0.9258],
        [-0.3126,  2.4152, -0.1106, -0.9931,  3.3449, -2.5229,  1.4187,  1.2196],
        [ 1.0876,  1.9652, -0.2621, -0.3158,  0.6091,  1.2616, -0.5484,  0.8048],
        [-1.8044, -0.4126, -0.8306,  0.5898, -0.7987, -0.5856,  0.6433,  0.6303]],
       grad_fn=<SelectBackward0>)

after causal masking:
tensor([[-1.7629,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf],
        [-3.3334, -1.6556,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf],
        [-1.0226, -1.2606,  0.0762,    -inf,    -inf,    -inf,    -inf,    -inf],
        [ 0.7836,

#### Scaled Dot-Product Attention

\begin{align}
\text{Attention}(Q,K,V) = \text{softmax}\left( \frac{QK^T}{\sqrt{d_{\mathrm{head}}}} \right)V
\end{align}

... we also need to scale that $Q K^{T}$ by $\frac{1}{\sqrt{d_{\mathrm{head}}}}$

In [15]:
k = torch.randn(B,T,head_size)
q = torch.randn(B,T,head_size)
wei = q @ k.transpose(-2,-1) #* head_size**-0.5

In [16]:
q.var()

tensor(1.0700)

In [17]:
k.var()

tensor(1.0449)

In [18]:
wei.var()

tensor(17.4690)

#### `wei`

In our implementation of self-attention, notice that `wei` is input to the softmax function.

To prevent underflow and overflow when using the softmax function, preventative measures can be taken.

And that is where that $\frac{1}{\sqrt{d_{\mathrm{head}}}}$ scaling factor comes in!

In [19]:
wei = q @ k.transpose(-2,-1) * head_size**-0.5
wei.var()

tensor(1.0918)

Without such measures `softmax(wei)` may tend towards a one-hot vector. The result of using a one-hot vector will be that attention will tend to be focused only on a single node in the past, and the other previous nodes will not be able to contribute to the prediction of the next token.

In [20]:
s = 16

torch.set_printoptions(precision=4)
print(torch.softmax(torch.tensor([0.1, -0.2, 0.3, 0.5])*s, dim=-1))
torch.set_printoptions(profile="default")

tensor([1.5939e-03, 1.3118e-05, 3.9103e-02, 9.5929e-01])


----

## Adding Single Head of Self-attention to `BigramLanguageModel` 

* Use token embeddings
* Use positional embeddings
* Implement self-attention per 3.2.1 Scaled Dot-product Attention, page 4 of [Attention Is All You Need](https://arxiv.org/pdf/1706.03762)
* Obtain the logits by feeding the product of self-attention weights with the input into a Linear layer

In [21]:
class Head(nn.Module):
    """ one head of self-attention """
    def __init__(self, head_size):
        super().__init__()

        self.key = nn.Linear(embedding_size, head_size, bias=False)
        self.query = nn.Linear(embedding_size, head_size, bias=False)
        self.value = nn.Linear(embedding_size, head_size, bias=False)

        self.register_buffer(
            'tril',
            torch.tril(torch.ones(max_context_size, max_context_size))
        )

    def forward(self, x):
        B,T,C = x.shape

        k = self.key(x)      # (B,T,C)
        q = self.query(x)    # (B,T,C)

        # compute weights for affinities (querys - keys)
        wei = q @ k.transpose(-2,-1) * C**-0.5   # (B,T,C) @ (B,C,T) ----> (B,T,T)

        # self-attention masking ensures tokens can only communicate with their past
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))     # (B,T,T)

        wei = F.softmax(wei, dim=-1)                                     # (B,T,T)

        # perform the weighted aggregation of the values across Time dimension
        v = self.value(x)    # (B,T,C)
        out = wei @ v        # (B,T,T) @ (B,T,C) ----> (B,T,C)

        return out

In [22]:
# improved simple bigram language model with self-attention
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.token_embedding_table = nn.Embedding(vocab_size, embedding_size)
        self.position_embedding_table = nn.Embedding(max_context_size, embedding_size)

        # define a single head of self-attention
        self.sa_head = Head(embedding_size)

        self.lm_head = nn.Linear(embedding_size, vocab_size)

    def forward(self, idx, targets=None):
        B,T = idx.shape

        # idx and targets are both (B,T) tensor of int
        tok_embeddings = self.token_embedding_table(idx)  # (B,T,C)
        pos_embeddings = self.position_embedding_table(torch.arange(T, device=device))  # (T,C)

        x = tok_embeddings + pos_embeddings  # (B,T,C)
        x = self.sa_head(x)                  # (B,T,C)
        logits = self.lm_head(x)             # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B,T) tensor of indices in the current context
        for _ in range(max_new_tokens):

            # crop idx to the last max_context_size tokens
            idx_cond = idx[:, -max_context_size:]

            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]           # becomes (B,C)
            probs = F.softmax(logits, dim=-1)   # (B,C)
            idx_next = torch.multinomial(probs, num_samples=1) # (B,1)
            idx = torch.cat((idx, idx_next), dim=1)  # (B,T+1)
        return idx

## The Training Loop

This is essentially the same as in `02_model_&_training-loop.ipynb`.

In [23]:
%%time

torch.manual_seed(1337)

model = BigramLanguageModel().to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(loss_estimation_iters)
        for k in range(loss_estimation_iters):
            X,Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

for iter in range(max_iters):

    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss is {losses['train']:.4f}, val loss is {losses['val']:.4f}")

    xb,yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(f"final loss: {loss.item():.4f}\n")

step 0: train loss is 4.2000, val loss is 4.2047
step 500: train loss is 2.6911, val loss is 2.7087
step 1000: train loss is 2.5196, val loss is 2.5303
step 1500: train loss is 2.4775, val loss is 2.4829
step 2000: train loss is 2.4408, val loss is 2.4523
step 2500: train loss is 2.4272, val loss is 2.4435
step 3000: train loss is 2.4130, val loss is 2.4327
step 3500: train loss is 2.3956, val loss is 2.4212
step 4000: train loss is 2.4041, val loss is 2.3992
step 4500: train loss is 2.3980, val loss is 2.4084
final loss: 2.5494

CPU times: user 12.2 s, sys: 334 ms, total: 12.6 s
Wall time: 12.9 s


In [24]:
# generate from the model
context = torch.zeros((1, 1), dtype=torch.int, device=device)
print(decode(
    model.generate(
        context,
        max_new_tokens=500
    )[0].tolist()
))


Whent iknt,
Thowi, ht son, bth

Hiset bobe ale.
S:
O-' st dalilanss:
Want he us he, vet?
Wedilas ate awice my.

HDET:
ANGo oug
Yowhavetof is he ot mil ndill, aes iree sen cie lat Herid ovets, and Win ngarigoerabous lelind peal.
-hule onchiry ptugr aiss hew ye wllinde norod atelaves
Momy yowod mothake ont-wou whth eiiby we ati dourive wee, ired thoouso er; th
To kad nteruptef so;
ARID Wam:
ENGCI inleront ffaf Pre?

Wh om.

He-
LIERCKENIGUICar adsal aces ard thinin cour ay aney Iry ts I fr af ve y


----

## Count the Model Weights

How many parameters does this model implementation have?

In [25]:
c = 0
for name, p in model.named_parameters():
    print(f"{name:60s} {p.numel():>10,}")
    c += p.numel()

print(f"{''.join(['-']*71)}")
print(f"{'total parameters':60s} {c:>10,}")

token_embedding_table.weight                                      2,080
position_embedding_table.weight                                     256
sa_head.key.weight                                                1,024
sa_head.query.weight                                              1,024
sa_head.value.weight                                              1,024
lm_head.weight                                                    2,080
lm_head.bias                                                         65
-----------------------------------------------------------------------
total parameters                                                  7,553
